## 读取数据

In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import gzip
import os
import struct
import numpy as np

def load_mnist(path, kind='train'):
    """加载MNIST数据集"""
    labels_path = os.path.join(path, f'{kind}-labels-idx1-ubyte.gz')
    images_path = os.path.join(path, f'{kind}-images-idx3-ubyte.gz')

    with gzip.open(labels_path, 'rb') as lbpath:
        struct.unpack('>II', lbpath.read(8))
        labels = np.frombuffer(lbpath.read(), dtype=np.uint8)

    with gzip.open(images_path, 'rb') as imgpath:
        struct.unpack('>IIII', imgpath.read(16))
        images = np.frombuffer(imgpath.read(), dtype=np.uint8).reshape(len(labels),1,28,28)

    return images, labels



# 数据集划分
def data_split(images, labels, ratio):
    
    total_len = images.shape[0]
    offset = int(total_len * ratio)

    val_img = images[:offset][:]
    val_lb = labels[:offset]

    train_img = images[offset:][:]
    train_lb = labels[offset:]

    return train_img, train_lb, val_img, val_lb    

# 读取训练集和测试集数据
[images, labels] = load_mnist('../实验课题库/MNIST', kind='train')
[test_img, test_lb] = load_mnist('../实验课题库/MNIST',kind='test')
train_img, train_lb, val_img, val_lb = data_split(images, labels, 1/6)

# 为了加快调试速度，从训练集选择2000个样本。
random_numbers = np.random.randint(50000, size=(2000, ))
train_img=train_img[random_numbers]
train_lb= train_lb [random_numbers]

# 将所有数据归一化到0-1之间
train_img =train_img/255.
val_img   =val_img/255.
test_img  =test_img/255.

# 对标签进行热编码
one_hot_train_lb = np.eye(10)[train_lb]
one_hot_val_lb = np.eye(10)[val_lb]
one_hot_test_lb= np.eye(10)[test_lb]

# 打印查看数据集格式
print('训练集图像格式为:', train_img.shape, '训练集标签格式为:', train_lb.shape,'热编码训练集标签格式为:', one_hot_train_lb.shape)
print('验证集图像格式为:', val_img.shape, '验证集标签格式为:', val_lb.shape,'热编码验证集标签格式为:', one_hot_val_lb.shape)
print('测试集图像格式为:', test_img.shape, '测试集标签格式为:', test_lb.shape,'热编码测试集标签格式为:', one_hot_test_lb.shape)

训练集图像格式为: (2000, 1, 28, 28) 训练集标签格式为: (2000,) 热编码训练集标签格式为: (2000, 10)
验证集图像格式为: (10000, 1, 28, 28) 验证集标签格式为: (10000,) 热编码验证集标签格式为: (10000, 10)
测试集图像格式为: (10000, 1, 28, 28) 测试集标签格式为: (10000,) 热编码测试集标签格式为: (10000, 10)


使用增强处理后的数据，训练网络

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels=1, out_channels=10, stride=3,padding=1,kernel_size=3)
        self.w1 =nn.Linear(10*10*10,100)
        self.w2 =nn.Linear(100,10)
        self.BN1=nn.BatchNorm2d(10)
        self.BN2=nn.BatchNorm1d(100)
        self.relu=nn.ReLU()


    def forward(self, x):
        x = self.conv1 (x)
        x = self.BN1(x)
        x = self.relu(x)
        
        x = x.view(x.size(0), -1)
        x = self.w1 (x)
        x = self.BN2(x)
        x = self.relu(x)

        x = self.w2 (x)       
        return x


In [11]:

model = NeuralNetwork()

# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

learning_rate = 5e-3
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

batch_size = 200
epochs = 100
batch_num=int(train_img.shape[0]/batch_size)
size = len(train_img)

model.train()
for t in range(epochs):
    
    correct=0.
    train_mean_loss=0.

    for batch in range(batch_num):
        X=train_img[batch*batch_size:(batch+1)*batch_size,]
        y=one_hot_train_lb[batch*batch_size:(batch+1)*batch_size,:]

        X=torch.tensor(X, dtype=torch.float32)
        y=torch.tensor(y, dtype=torch.float32)
        
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        correct += (pred.argmax(1) == y.argmax(1)).type(torch.float).mean().item()
        train_mean_loss+= loss.item()

    train_mean_loss /= batch_num
    correct /= batch_num
    
    print(f" Epoch:{t}, loss: {train_mean_loss:>8f},  Accuracy: {(100*correct):>0.1f}%")


 Epoch:0, loss: 2.253637,  Accuracy: 16.3%
 Epoch:1, loss: 1.996825,  Accuracy: 38.7%
 Epoch:2, loss: 1.792803,  Accuracy: 55.5%
 Epoch:3, loss: 1.636129,  Accuracy: 65.4%
 Epoch:4, loss: 1.514137,  Accuracy: 70.9%
 Epoch:5, loss: 1.416251,  Accuracy: 74.1%
 Epoch:6, loss: 1.335202,  Accuracy: 76.4%
 Epoch:7, loss: 1.266967,  Accuracy: 77.8%
 Epoch:8, loss: 1.208394,  Accuracy: 79.2%
 Epoch:9, loss: 1.157241,  Accuracy: 80.1%
 Epoch:10, loss: 1.111846,  Accuracy: 80.6%
 Epoch:11, loss: 1.071085,  Accuracy: 81.2%
 Epoch:12, loss: 1.034131,  Accuracy: 81.8%
 Epoch:13, loss: 1.000327,  Accuracy: 82.6%
 Epoch:14, loss: 0.969129,  Accuracy: 83.2%
 Epoch:15, loss: 0.940217,  Accuracy: 83.7%
 Epoch:16, loss: 0.913252,  Accuracy: 84.0%
 Epoch:17, loss: 0.888069,  Accuracy: 84.5%
 Epoch:18, loss: 0.864469,  Accuracy: 85.2%
 Epoch:19, loss: 0.842272,  Accuracy: 85.7%
 Epoch:20, loss: 0.821347,  Accuracy: 86.1%
 Epoch:21, loss: 0.801543,  Accuracy: 86.3%
 Epoch:22, loss: 0.782790,  Accuracy: 87.0

In [11]:
torch.save(model.state_dict(), 'model_weights.pth')

In [12]:
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))

<All keys matched successfully>

In [10]:
torch.save(model, 'model.pth')

In [7]:
model = torch.load('model.pth', weights_only=False)

In [9]:
model.eval()
test_loss, correct = 0, 0
loss_fn = nn.CrossEntropyLoss()
with torch.no_grad():
        X=torch.tensor(test_img, dtype=torch.float32)
        y=torch.tensor(one_hot_test_lb, dtype=torch.float32)
        pred = model(X)
        test_loss = np.mean(loss_fn(pred, y).item())
        correct = (pred.argmax(1) == y.argmax(1)).type(torch.float).mean().item()
print(f"Test Accuracy: {(100*correct):>0.1f}%, Test Avg loss: {test_loss:>8f} \n")

Test Accuracy: 90.0%, Test Avg loss: 0.413730 

